In [2]:
import pandas as pd

new_data = pd.read_csv(
    "/Users/littleair/Desktop/ansci-4040-fall-2026-Lab/Project_1/data/PracticeTest.csv",
    na_values=["null"]
)

new_data.columns = new_data.columns.str.strip()

print(new_data.columns.tolist())
print(new_data.dtypes)
print(new_data.isna().sum())

['AnimalId', 'LactationNumber', 'DaysInMilk', 'ReproductionStatus', 'EventDate', 'Avgmilkflow', 'Flow30_60Session', 'YieldFirst2Min_Session', 'YieldSession', 'DurationSession_sec', 'milking', 'MilkingId']
AnimalId                  float64
LactationNumber           float64
DaysInMilk                float64
ReproductionStatus        float64
EventDate                     str
Avgmilkflow               float64
Flow30_60Session          float64
YieldFirst2Min_Session    float64
YieldSession              float64
DurationSession_sec         int64
milking                     int64
MilkingId                   int64
dtype: object
AnimalId                  1701003
LactationNumber           1701003
DaysInMilk                1701003
ReproductionStatus        1701003
EventDate                       0
Avgmilkflow                    29
Flow30_60Session                0
YieldFirst2Min_Session          0
YieldSession                    0
DurationSession_sec             0
milking                         0

In [3]:
import joblib

knn_model = joblib.load("knn_model.joblib")

print(knn_model)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformer_weights={'calving': 512.0,
                                                        'lactation': 8.0,
                                                        'status': 0.25},
                                   transformers=[('numeric', StandardScaler(),
                                                  ['flow_30_60_session',
                                                   'yield_first_2min_session',
                                                   'yield_session',
                                                   'duration_session_sec',
                                                   'milking', 'avg_milk_flow',
                                                   'days_in_milk',
                                                   'event_month',
                                                   'event_day_of_year',
                                                   'event_weekday',
               

In [5]:
import pandas as pd

measurement_cols = [
    "flow_30_60_session", "yield_first_2min_session",
    "yield_session", "duration_session_sec", "milking",
    "avg_milk_flow", "days_in_milk", "lactation_number"
]

def build_features(records):
    X = records[measurement_cols].astype("float32").copy()
    dates = pd.to_datetime(records["event_date"], errors="raise")

    X["estimated_calving_day"] = (
        (dates - pd.Timestamp("2000-01-01")).dt.days
        - X["days_in_milk"]
    )

    X["event_month"] = dates.dt.month
    X["event_day_of_year"] = dates.dt.dayofyear
    X["event_weekday"] = dates.dt.dayofweek
    X["event_year"] = dates.dt.year
    X["reproduction_status"] = records["reproduction_status"]

    return X

In [7]:
new_data.columns = new_data.columns.str.strip()

new_data = new_data.rename(columns={
    "AnimalId": "animal_id",
    "LactationNumber": "lactation_number",
    "DaysInMilk": "days_in_milk",
    "ReproductionStatus": "reproduction_status",
    "EventDate": "event_date",
    "Avgmilkflow": "avg_milk_flow",
    "Flow30_60Session": "flow_30_60_session",
    "YieldFirst2Min_Session": "yield_first_2min_session",
    "YieldSession": "yield_session",
    "DurationSession_sec": "duration_session_sec",
    "MilkingId": "milking_id"
})

In [10]:
X_new = build_features(new_data.iloc[:5])

X_new["reproduction_status"] = (
    X_new["reproduction_status"].astype(object)
)

predicted_ids = knn_model.predict(X_new)

print(predicted_ids)

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Reflection
I applied my saved KNN model to the first five records in the new dataset without changing the model or filling in missing values. However, prediction failed because the KNN classifier does not accept NaN values. I realized that I had included days_in_milk and lactation_number when training my original model, even though these variables are unavailable in the records that need Animal ID prediction. Without “days_in_milk” and missing “event_date”, I also cannot calculate "estimated calving day", which was a key feature in my model.

My current model is not suitable for the intended Animal ID prediction. Everything I did in the model training is to train a model that is suitable for the training set only. The reason for that is I only focused on improving the accuracy rather than thinking what it would be like in the real world situation. So I persistently used the variables that are missing with Animal ID in the real dataset. Therefore, my validation and test set did not represent the actual prediction conditions.

If I can revise my model, I would remove days_in_milk, lactation_number, reproduction_status, and estimated_calving_day, and retrain KNN using only information available in the records. My hyperparameter tuning process would involve comparing different parameter combinations and then evaluating different feature weights.

A good model should be designed around its intended use, not overfitting to the training set.  Most importantly, practical real-world scenarios must be taken into account.